# B08-02 — Gaussian Process·Expected Improvement 기반 회고적 Bayesian Optimization

**연구 질문:** 동일한 초기 관측 집합과 추가 선택 예산에서 Random, RF_greedy, GP-EI가 실제 측정값이 있는 후보 중 더 높은 `K_median` 또는 `r_median`을 얼마나 빨리 찾는가?

- **실제 실험을 새로 하는 것이 아닙니다.** 기존 1,025개 배지 조성의 측정값을 선택 전까지 숨겼다가 공개하는 회고적 시뮬레이션입니다.
- Bayesian Optimization(BO)은 Gaussian Process(GP) surrogate와 Expected Improvement(EI) 획득함수를 사용합니다.
- B07에서 실험 라운드 보류 일반화가 크게 실패한 사례가 있으므로 이번 성과를 전향적 실험 성과로 해석하지 않습니다.
- 원본 논문의 `K`, `r` 정의를 확정하지 않은 상태이므로 조성 수준 요약 목표 `K_median`, `r_median` 그대로 사용합니다.

**선행 실습:** B08-00 Clean v2, B08-01 v1. 실행 커널: `Python (BioImage AI)`.

## 1. 실행 환경과 기존 산출물 연결

입력은 B08-00의 `candidate_pool_measured_profiles.csv`, B07-00의 `medium_feature_audit.csv`입니다. B08-01은 재현 기준값을 비교하기 위한 입력으로만 읽고, 그 결과를 GP 모델 학습에 사용하지 않습니다.

In [ ]:
from pathlib import Path
import json, hashlib, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.special import ndtr

cwd=Path.cwd().resolve()
ROOT=next((p for p in [cwd,*cwd.parents] if (p/'results'/'bioculture'/'B08_00_candidate_design_v2').is_dir()),None)
if ROOT is None:
    raise FileNotFoundError('B08-00 Clean v2 산출물이 있는 bioimage-ai 프로젝트 아래에서 실행하세요.')
BASE=ROOT/'results'/'bioculture'
CAND=BASE/'B08_00_candidate_design_v2'/'candidate_pool_measured_profiles.csv'
FEATURE=BASE/'B07_00_ml_data_audit_v2'/'medium_feature_audit.csv'
PRIOR=BASE/'B08_01_sequential_selection_v1'/'sequential_selection_summary.csv'
OUT=BASE/'B08_02_bayesian_optimization_v1';OUT.mkdir(parents=True,exist_ok=True)
SEEDS=[17,43,101]; N_INITIAL=30; N_STEPS=20; N_TREES=100
TARGETS=['K_median','r_median']; STRATEGIES=['Random','RF_greedy','GP_EI']
PCA_DIM=5; GP_NOISE=1e-3; EI_XI=0.01
for p in [CAND,FEATURE]:
    if not p.is_file(): raise FileNotFoundError(p)
print('Python:',sys.version.split()[0]);print('Output:',OUT)
print('GP: Matern(ν=2.5), fixed kernel, PCA dims:',PCA_DIM)

## 2. 후보와 입력 성분 감사

B08-01과 정확히 같은 후보 포함 기준(`target` 비결측)과 같은 초기화 시드를 적용합니다. 44개 배지 성분의 값만을 입력 변수로 사용하고 목표 열이 섞이지 않았는지 검사합니다.

In [ ]:
df=pd.read_csv(CAND,encoding='utf-8-sig',dtype={'medium_profile_id':'string'},low_memory=False)
feature_info=pd.read_csv(FEATURE,encoding='utf-8-sig')
features=feature_info['feature'].astype(str).tolist()
assert len(df)==1025 and df.medium_profile_id.is_unique
assert len(features)==44 and len(set(features))==44
assert set(features).issubset(df.columns)
assert set(TARGETS).issubset(df.columns)
assert not set(features).intersection({'medium_profile_id','Condition ID','Assay ID',*TARGETS})
assert all(pd.to_numeric(df[c],errors='coerce').notna().all() for c in features)
assert np.isfinite(df[features].to_numpy(dtype=float)).all()
assert (df[features].to_numpy(dtype=float)>=0).all()
for t in TARGETS: df[t]=pd.to_numeric(df[t],errors='coerce')
print('조성:',len(df),'입력 변수:',len(features))
print('목표별 유효 조성:',{t:int(df[t].notna().sum()) for t in TARGETS})

## 3. 관측값 격리와 선택 함수

`HiddenOracle`은 모든 실측값을 보관하지만 후보 선택 함수에는 **이미 공개한 값만** 넘깁니다. GP는 배지 성분 44차원을 표준화·PCA(5차원)한 **X만**으로 후보 위치를 표현합니다. 후보의 X 전체가 미리 알려져 있다는 오프라인 후보군 가정하에서만 허용되는 전처리입니다. GP의 목표값은 언제나 공개된 관측값만입니다.

계산 시간·수치 안정성을 위해 GP 하이퍼파라미터는 매 회 재최적화하지 않고 고정합니다. 이는 간단한 교육용 BO이지 최적의 GP 구현을 주장하지 않습니다.

In [ ]:
class HiddenOracle:
    def __init__(self, ids, values):
        self.__values=dict(zip([str(x) for x in ids],np.asarray(values,dtype=float)))
        self.revealed=set()
    def reveal(self, uid):
        uid=str(uid)
        if uid in self.revealed: raise AssertionError('동일 조성을 중복 공개하려고 했습니다.')
        val=float(self.__values[uid]);self.revealed.add(uid);return val

def rf_select(known_x,known_y,unknown_x,seed):
    model=Pipeline([('imputer',SimpleImputer(strategy='median')),
       ('rf',RandomForestRegressor(n_estimators=N_TREES,max_features=0.8,
                min_samples_leaf=2,n_jobs=-1,random_state=int(seed)))])
    model.fit(known_x,known_y)
    scores=model.predict(unknown_x)
    return int(np.argmax(scores)),float(np.max(scores)),np.nan

def expected_improvement(mu,sigma,best,xi=EI_XI):
    mu=np.asarray(mu,dtype=float);sigma=np.asarray(sigma,dtype=float)
    imp=mu-best-xi
    safe=np.maximum(sigma,1e-12)
    z=imp/safe
    # EI = (μ-best-ξ) Φ(z) + σ φ(z), zero at effectively no uncertainty.
    from scipy.stats import norm
    ei=imp*ndtr(z)+safe*norm.pdf(z)
    return np.where(sigma>1e-12,np.maximum(ei,0),0.0)

def gp_ei_select(known_x,known_y,unknown_x,best):
    y=np.asarray(known_y,dtype=float)
    # Target scaling preserves the maximizer; scales EI_XI to the observed target spread.
    ym=float(y.mean());ys=float(y.std())
    if ys<1e-10: ys=1.0
    kernel=ConstantKernel(1.0,constant_value_bounds='fixed')*Matern(length_scale=2.0,
                              length_scale_bounds='fixed',nu=2.5)
    gp=GaussianProcessRegressor(kernel=kernel,alpha=GP_NOISE,
                                 optimizer=None,normalize_y=False,random_state=0)
    gp.fit(known_x,(y-ym)/ys)
    mu_z,sigma_z=gp.predict(unknown_x,return_std=True)
    ei=expected_improvement(mu_z,sigma_z,(best-ym)/ys)
    j=int(np.argmax(ei))
    return j,float(mu_z[j]*ys+ym),float(sigma_z[j]*ys)

# 수학적 기본 테스트: 예측 분산이 0이면 EI도 0
assert np.all(expected_improvement([0,1],[0,0],0.5)==0)
assert expected_improvement([2],[0.2],1)[0]>expected_improvement([0],[0.2],1)[0]
print('오라클·EI 함수 검사 통과')

## 4. 회고적 순차 실험: 세 전략 비교

각 목표·시드마다 초기 30개 조성을 **정확히 같은 ID 집합**으로 선택합니다. 그 뒤 추가 20회 선택합니다. 초기 무작위 순열은 B08-01과 동일하게 `default_rng(seed).permutation`이며 RF 매 단계 난수 시드는 `seed+step`입니다. 같은 후보 ID에서 동점이 생기면 정렬된 ID 순서로 결정합니다.

PCA는 알려진 후보의 **X 정보만** 이용해 한 번 수행하며 목표값은 PCA·표준화에 입력하지 않습니다. 실험적 배포에선 측정 전 알 수 없는 신규 후보가 있다면 별도 재설계가 필요합니다.

In [ ]:
def simulate_target(table,target,seed,strategy):
    sub=table.loc[table[target].notna(),['medium_profile_id',*features,target]].copy()
    sub=sub.sort_values('medium_profile_id').reset_index(drop=True)
    ids=sub.medium_profile_id.astype(str).to_numpy()
    X=sub[features].to_numpy(dtype=float)
    oracle=HiddenOracle(ids,sub[target].to_numpy(dtype=float))
    perm=np.random.default_rng(seed).permutation(len(ids))
    initial=perm[:N_INITIAL]
    # X-only transformation, no target values used.
    if strategy=='GP_EI':
        Xscaled=StandardScaler().fit_transform(X)
        Xgp=PCA(n_components=min(PCA_DIM,X.shape[1]),svd_solver='full').fit_transform(Xscaled)
        Xgp=StandardScaler().fit_transform(Xgp)
    revealed=[];seen=set()
    for j in initial:
        uid=str(ids[j]); val=oracle.reveal(uid)
        seen.add(uid);revealed.append((uid,val))
    initial_best=max(v for _,v in revealed);best=initial_best
    chooser_rng=np.random.default_rng(seed+100_000)
    rows=[]
    for step in range(1,N_STEPS+1):
        known_mask=np.isin(ids,list(seen));unknown_mask=~known_mask
        known_ids=ids[known_mask];unknown_ids=ids[unknown_mask]
        known_y=np.array([dict(revealed)[str(uid)] for uid in known_ids],dtype=float)
        assert len(unknown_ids)==len(ids)-N_INITIAL-step+1
        pred=np.nan;pred_std=np.nan;acq=np.nan
        if strategy=='Random':
            j=int(chooser_rng.integers(0,len(unknown_ids)))
        elif strategy=='RF_greedy':
            j,pred,pred_std=rf_select(X[known_mask],known_y,X[unknown_mask],seed+step)
        elif strategy=='GP_EI':
            j,pred,pred_std=gp_ei_select(Xgp[known_mask],known_y,Xgp[unknown_mask],best)
        else: raise ValueError(strategy)
        uid=str(unknown_ids[j]);assert uid not in seen
        value=oracle.reveal(uid);seen.add(uid);revealed.append((uid,value))
        best=max(best,value)
        rows.append({'target':target,'seed':seed,'strategy':strategy,'step':step,
                     'chosen_medium_profile_id':uid,'revealed_value':value,
                     'predicted_before_reveal':pred,'predictive_std':pred_std,
                     'best_so_far':best,'initial_best':initial_best,
                     'n_revealed':N_INITIAL+step})
    assert len(seen)==N_INITIAL+N_STEPS and len(oracle.revealed)==len(seen)
    return rows

rows=[]
for target in TARGETS:
    for seed in SEEDS:
        for strategy in STRATEGIES:
            run=simulate_target(df,target,seed,strategy)
            rows.extend(run)
            print(target,seed,strategy,'initial',round(run[0]['initial_best'],4),
                  'final',round(run[-1]['best_so_far'],4))
traces=pd.DataFrame(rows)
assert len(traces)==len(TARGETS)*len(SEEDS)*len(STRATEGIES)*N_STEPS
assert traces.groupby(['target','seed','strategy']).size().eq(N_STEPS).all()
assert traces.groupby(['target','seed'])['initial_best'].nunique().eq(1).all()
assert traces.groupby(['target','seed','strategy']).chosen_medium_profile_id.nunique().eq(N_STEPS).all()
assert traces.groupby(['target','seed','strategy']).best_so_far.apply(lambda z:(z.diff().fillna(0)>=-1e-12).all()).all()
print('총 선택 기록:',len(traces))

## 5. 성과 지표와 B08-01 비교 감사

- 최종 최고 발견값(`final_best`): 20회 선택 후 최고 공개값.
- 초기 대비 개선(`gain_from_initial`): 초기 최고값과의 차이.
- 발견 속도 지표(`mean_best_over_steps`): 20개 단계의 `best_so_far` 평균.
- 독립 실험 반복이 아니라 동일 역사적 데이터의 서로 다른 3개 초기화 시나리오입니다.

B08-01의 Random·RF_greedy 결과 파일이 있으면 **같은 설정의 동작을 재현했는지** 비교합니다. 이 비교는 결과 불일치 시 진행을 차단합니다.

In [ ]:
summary=(traces.sort_values('step').groupby(['target','seed','strategy'],as_index=False)
         .tail(1)[['target','seed','strategy','initial_best','best_so_far','n_revealed']]
         .rename(columns={'best_so_far':'final_best'}))
summary['gain_from_initial']=summary.final_best-summary.initial_best
area=(traces.groupby(['target','seed','strategy'],as_index=False)
       .agg(mean_best_over_steps=('best_so_far','mean')))
summary=summary.merge(area,on=['target','seed','strategy'],validate='one_to_one')
assert (summary.gain_from_initial>=-1e-12).all()
print(summary.to_string(index=False))
print('\n시드별 반복 평균 및 표준편차 (독립 생물학적 반복이 아님):')
print(summary.groupby(['target','strategy'])[['final_best','gain_from_initial','mean_best_over_steps']]
     .agg(['mean','std']).round(4).to_string())
prior_check={'present':bool(PRIOR.is_file()),'checked':False}
if PRIOR.is_file():
    old=pd.read_csv(PRIOR,encoding='utf-8-sig')
    check=summary[summary.strategy.isin(['Random','RF_greedy'])].merge(
        old,on=['target','seed','strategy'],suffixes=('_new','_old'),validate='one_to_one')
    assert len(check)==2*len(SEEDS)*len(TARGETS),'기존 B08-01과 비교 행 수 불일치'
    for c in ['initial_best','final_best','gain_from_initial','mean_best_over_steps']:
        assert np.allclose(check[f'{c}_new'],check[f'{c}_old'],atol=1e-10),f'B08-01과 {c} 불일치'
    prior_check.update({'checked':True,'matched_runs':len(check)})
    print('B08-01 Random/RF 재현 비교: 통과')
else:
    print('B08-01 기준 결과 파일 없음 — 교차 비교 생략 (B08-02 자체 감사는 계속)')

## 6. 시각화 및 라운드 한계

표시는 관측된 최고 성장 지표의 회고적 누적 경로입니다. **모델의 외부 실험 성공 확률이나 보정된 예측구간이 아닙니다.** GP 표준편차는 고정된 커널·단순 잡음 모델에 의존합니다.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.2),constrained_layout=True)
for ax,target in zip(axes,TARGETS):
    ss=traces.loc[traces.target.eq(target)]
    for strategy in STRATEGIES:
        a=ss.loc[ss.strategy.eq(strategy)].groupby('step').best_so_far.agg(['mean','std'])
        ax.plot(a.index,a['mean'],marker='o',markersize=2.2,label=strategy)
        ax.fill_between(a.index,(a['mean']-a['std']).to_numpy(),(a['mean']+a['std']).to_numpy(),alpha=.12)
    ax.set(title=f'{target}: retrospective discovery',xlabel='Additional measured profiles',ylabel='Best observed target')
    ax.grid(alpha=.25);ax.legend(fontsize=8)
path_fig=OUT/'retrospective_bo_comparison.png'
fig.savefig(path_fig,dpi=150,bbox_inches='tight');plt.show()
print('Figure:',path_fig.name)

## 7. 산출물 저장 및 재현성·누출 감사

시드·전략·선택 순서·공개값을 모두 저장합니다. SHA256은 실제 출력 파일 바이트의 동일성을 식별할 수 있지만 과학적 타당성을 독립적으로 입증하지는 않습니다.

In [ ]:
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

p_trace=OUT/'bayesian_sequential_selection_trace.csv'
p_summary=OUT/'bayesian_sequential_selection_summary.csv'
p_report=OUT/'B08_02_validation_report.json'
traces.to_csv(p_trace,index=False,encoding='utf-8-sig')
summary.to_csv(p_summary,index=False,encoding='utf-8-sig')
report={'status':'RETROSPECTIVE_BO_EVALUATED_NOT_NEW_EXPERIMENT',
    'n_all_profiles':int(len(df)),'n_features':len(features),
    'valid_profile_counts':{t:int(df[t].notna().sum()) for t in TARGETS},
    'targets':TARGETS,'seeds':SEEDS,'n_initial':N_INITIAL,'n_additional_steps':N_STEPS,
    'strategies':STRATEGIES,'pca_dimensions':PCA_DIM,'gp_alpha':GP_NOISE,'ei_xi':EI_XI,
    'gp_kernel':'Fixed ConstantKernel * Matern(nu=2.5, length_scale=2.0)',
    'pca_uses_all_candidate_X_only':True,'oracle_isolation':'Only revealed target values sent to selectors',
    'B08_01_reproduction':prior_check,
    'claims_not_supported':['new biological experiments','causal nutrient effects',
      'external generalization','calibrated GP predictive intervals','GP kernel optimality'],
    'target_definitions_verified':False}
p_report.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
a=pd.read_csv(p_trace,encoding='utf-8-sig');b=pd.read_csv(p_summary,encoding='utf-8-sig')
assert len(a)==len(traces) and len(b)==len(summary)
assert a.groupby(['target','seed','strategy']).size().eq(N_STEPS).all()
assert not a[['target','seed','strategy','step']].duplicated().any()
assert a.groupby(['target','seed','strategy']).chosen_medium_profile_id.nunique().eq(N_STEPS).all()
files=[CAND,FEATURE,*([PRIOR] if PRIOR.is_file() else []),p_trace,p_summary,p_report,path_fig]
manifest=pd.DataFrame([{'path':str(p.relative_to(ROOT)),'bytes':p.stat().st_size,'sha256':sha256(p),
                        'role':'input' if p in [CAND,FEATURE,PRIOR] else 'output'} for p in files])
manifest.to_csv(OUT/'input_output_sha256_manifest.csv',index=False,encoding='utf-8-sig')
print('검증 통과:',report['status']);print('저장 위치:',OUT)

## 8. 과학적 해석 및 연습문제

- BO가 이긴 경우에도 동일 데이터셋의 회고적 사례에 한정됩니다. B07에서 확인된 Round 02·05의 실패 및 라운드 간 분포 차이는 해결되지 않았습니다.
- 여기서 GP 표준편차는 **모델 기반 예측 불확실성**이며 실험실 측정 오차나 신규 라운드 편향을 모두 담은 구간이 아닙니다.
- `PCA`는 전체 후보의 입력 성분만 사용했습니다. 목표값 누출은 없지만 아직 측정하지 않은 배지 조성의 X를 모두 알고 있다는 **고정 후보군** 가정입니다.
- 어떤 조성도 관측 결과가 없는 미래 후보까지 추천하거나 실제 최적 배지라고 주장하지 않습니다.

### 연습문제 (풀이·Python 코드는 별도 Word 해설서에 수록)

1. GP 기반 BO에서 예측 평균과 예측 표준편차가 각기 어떤 역할을 하는지 설명해 보세요.
2. EI의 탐색(exploration)과 활용(exploitation) 사이 균형을 `EI_XI`를 바꿔 비교해 보세요. 왜 더 높은 `EI_XI`가 항상 좋은 결과를 보장하지 않을까요?
3. `SEEDS`를 5개로 확장해 세 전략의 최종 최고값 평균과 분산을 비교해 보세요. 이를 독립 실험 신뢰구간으로 부르지 않는 이유는 무엇인가요?
4. `PCA_DIM`을 3, 5, 8로 바꿔 GP_EI 결과가 어떻게 달라지는지 비교해 보세요. 목표값을 PCA 입력에 넣으면 왜 안 되나요?
5. Round 05와 같은 새 실험 라운드에서 본 Notebook의 성과가 그대로 유지된다고 말할 수 없는 이유를 설명해 보세요. 새로운 검증 프로토콜도 제안해 보세요.